# Python编程基础 · 工单1.3《工业故障分析报告生成系统》 — 教师参考版
第9周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/02-程序设计基础


## 参考实现 `w09`（含依赖导入）

In [2]:
# -*- coding: utf-8 -*-
"""W09 工业故障分析报告生成系统 —— jieba分词 + Counter词频 + bar.png + cloud.png"""
import json
import os
import random
import re
import sys
from collections import Counter

import numpy as np  # 通用科学计算
import matplotlib
matplotlib.use("Agg")          # 无显示环境同样出图
import matplotlib.pyplot as plt
from matplotlib import cm, font_manager

# ---- 中文CJK字体: 复用资料包 fonts/ZH_Font.ttf(与W10《质量日报》同一份字体) ----
_FONT_CANDIDATES = ["fonts/ZH_Font.ttf", "fonts/SimHei.ttf"]
_FONT_FALLBACK = ["/System/Library/Fonts/Supplemental/Arial Unicode MS.ttf",
                  "/System/Library/Fonts/PingFang.ttc",
                  "/System/Library/Fonts/Supplemental/Songti.ttc",
                  "/System/Library/Fonts/STHeiti Light.ttc"]


def find_cjk_font():
    """按优先级定位CJK字体, 返回绝对路径; 找不到返回None"""
    for p in _FONT_CANDIDATES:
        if os.path.exists(p) and os.path.getsize(p) > 100_000:
            return os.path.abspath(p)
    import glob
    for root in (os.getcwd(), os.path.dirname(os.getcwd())):
        for p in sorted(glob.glob(os.path.join(root, "**", "fonts", "*.ttf"),
                                  recursive=True)):
            if os.path.getsize(p) > 100_000:
                return os.path.abspath(p)
    for p in _FONT_FALLBACK:
        if os.path.exists(p):
            return p
    return None


def setup_cjk_font():
    """注册CJK字体并配好rcParams, 返回(字体绝对路径, 字体族名)"""
    path = find_cjk_font()
    family = "Arial Unicode MS"
    if path:
        try:
            font_manager.fontManager.addfont(path)
            family = font_manager.FontProperties(fname=path).get_name()
        except Exception as exc:                     # 注册失败也不能让中文变方块
            print("[w09] 字体注册失败, 退回候选族名:", exc)
    matplotlib.rcParams["font.sans-serif"] = [family, "SimHei", "Arial Unicode MS"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    return path, family


def w09():
    import jieba
    import jieba.analyse

    # ---- 语料: 与教师参考 solutions.w09() 完全一致(6条故障描述 × 3 = 18条) ----
    texts = ["1号注塑机液压油温升高报警", "3号车床主轴轴承磨损异响",
             "空压机排气压力不足", "PLC通信中断从站掉线",
             "冷却水温持续偏高填料结垢", "变频器过流报警负载卡滞"] * 3
    stop = set("的 了 在 是 请 尽快 影响 报警 需".split())

    # TODO-1 分词 + 停用词过滤 → words列表
    words = [w for t in texts for w in jieba.cut(t) if w not in stop and len(w) > 1]

    # TODO-2 词频统计(Counter) + Top10柱状图(保存bar.png)
    top10 = Counter(words).most_common(10)
    print("Top10:", top10)
    print("Top10 词频表:")
    for rank, (word, cnt) in enumerate(top10, 1):
        print(f"  {rank:>2}. {word:<6} {cnt}")

    # TODO-3 jieba.analyse.extract_tags(text, topK=20, withWeight=True) 关键词
    print("关键词:", jieba.analyse.extract_tags(" ".join(texts), topK=8))
    print("关键词(含权重):")
    for kw, wt in jieba.analyse.extract_tags(" ".join(texts), topK=5,
                                             withWeight=True):
        print(f"  {kw:<6} {wt:.4f}")

    font_path, family = setup_cjk_font()
    print(f"中文字体文件: {font_path}")
    print(f"字体族名(font.sans-serif): {family}")

    # ---- TODO-2: Top10 水平条形图 → bar.png ----
    labels = [w for w, _ in top10][::-1]
    counts = [c for _, c in top10][::-1]
    fig, ax = plt.subplots(figsize=(7, 4.2), dpi=120)
    bars = ax.barh(labels, counts, color="#2b6cb0")
    ax.set_title("工业故障描述 Top10 关键词词频")
    ax.set_xlabel("词频")
    ax.set_xlim(0, max(counts) * 1.2)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_width() + max(counts) * 0.02,
                bar.get_y() + bar.get_height() / 2, str(cnt),
                va="center", fontsize=9)
    fig.tight_layout()
    fig.savefig("bar.png")
    plt.close(fig)

    # ---- TODO-4: wordcloud 生成词云 → cloud.png ----
    engine = ""
    try:
        from wordcloud import WordCloud
        wc = WordCloud(font_path=font_path, width=900, height=450,
                       background_color="white", random_state=42).generate(
                           " ".join(words))
        wc.to_file("cloud.png")
        engine = f"wordcloud(font_path={font_path})"
    except Exception as exc:
        # 零依赖兜底(多解法手册口径): matplotlib 等比"标签云", 字号∝词频, 不做重排
        print("[w09] wordcloud 不可用 → matplotlib 标签云零依赖兜底:", exc)
        rnd = np.random.default_rng(7)
        fig, ax = plt.subplots(figsize=(9, 4.5), dpi=120)
        ax.set_axis_off()
        ys = [0.85, 0.63, 0.41, 0.19]
        for i, (word, cnt) in enumerate(top10):
            ax.text(0.05 + (i % 5) * 0.19, ys[i // 5] + 0.02 * rnd.random(),
                    f"{word}×{cnt}", fontsize=10 + 14 * cnt / max(counts),
                    color=cm.tab10(i % 10), transform=ax.transAxes)
        ax.set_title("工业故障关键词标签云(matplotlib零依赖兜底, 字号∝词频)")
        fig.tight_layout()
        fig.savefig("cloud.png")
        plt.close(fig)
        engine = "matplotlib标签云兜底(零依赖, 字号∝词频)"

    # ---- 交付物: 路径 + 字节数 ----
    print(f"词云实现: {engine}")
    print("已保存 bar.png :", os.path.abspath("bar.png"),
          os.path.getsize("bar.png"), "bytes")
    print("已保存 cloud.png:", os.path.abspath("cloud.png"),
          os.path.getsize("cloud.png"), "bytes")

    # ---- 核验: PIL 打开, 确认非空且 >1KB ----
    from PIL import Image
    for name in ("bar.png", "cloud.png"):
        size = os.path.getsize(name)
        with Image.open(name) as im:
            width, height, mode = im.size[0], im.size[1], im.mode
        verdict = "通过" if size > 1024 and width > 100 else "异常"
        print(f"核验 {name}: {size} bytes, {width}x{height}px, {mode} → {verdict}")


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `1` | TODO-1 分词+停用词过滤 → words列表 |
| `2` | TODO-2 词频统计(Counter) + Top10柱状图(保存bar.png) TODO-3 |
|  | jieba.analyse.extract_tags(text, topK=20, withWeight=True) |
|  | 关键词 TODO-4 wordcloud生成词云(保存cloud.png): |
| `5` | TODO-5 (洞察)打印Top10词, 撰写≤150字故障规律分析(写入工单) |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 多解对比（已实跑互验）
| 解 | 路线 | 结论 |
|----|------|------|
| A | jieba精确切+Counter | 计数直白 |
| B | jieba TF-IDF关键词 | 自动去高频停用（各Top5） |

> 决策素材：词云要全量词频→A；报告要关键词→B。

## 执行与验证

In [3]:
w09()

Building prefix dict from the default dictionary ...


Loading model from cache /var/folders/6j/khvhkzp55m30fx539sm9tggr0000gn/T/jieba.cache


Loading model cost 0.336 seconds.


Prefix dict has been built successfully.


Top10: [('注塑机', 3), ('液压油', 3), ('温升高', 3), ('车床', 3), ('主轴', 3), ('轴承', 3), ('磨损', 3), ('异响', 3), ('空压机', 3), ('排气', 3)]
Top10 词频表:
   1. 注塑机    3
   2. 液压油    3
   3. 温升高    3
   4. 车床     3
   5. 主轴     3
   6. 轴承     3
   7. 磨损     3
   8. 异响     3
   9. 空压机    3
  10. 排气     3
关键词: ['报警', '注塑机', '掉线', '温升高', '空压机', '液压油', '异响', 'PLC']
关键词(含权重):
  报警     0.5977
  注塑机    0.4965
  掉线     0.4965
  温升高    0.4390
  空压机    0.4390
中文字体文件: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/02-程序设计基础/fonts/ZH_Font.ttf
字体族名(font.sans-serif): Arial Unicode MS


词云实现: wordcloud(font_path=/Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/02-程序设计基础/fonts/ZH_Font.ttf)
已保存 bar.png : /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/02-程序设计基础/bar.png 29761 bytes
已保存 cloud.png: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/02-程序设计基础/cloud.png 81770 bytes
核验 bar.png: 29761 bytes, 840x504px, RGBA → 通过
核验 cloud.png: 81770 bytes, 900x450px, RGB → 通过


## 交付物核验（PIL 独立复算）

In [4]:
# --- 独立复算: 不依赖 w09() 内部状态, 重新核对两张图 ---
import os
from PIL import Image

for name in ("bar.png", "cloud.png"):
    path = os.path.abspath(name)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else -1
    if exists:
        with Image.open(path) as im:
            dims, mode = im.size, im.mode
    else:
        dims, mode = (0, 0), "-"
    ok = exists and size > 1024 and dims[0] > 100 and dims[1] > 100
    print(f"{name}: {'PASS' if ok else 'FAIL'} | path={path} | "
          f"{size} bytes | {dims[0]}x{dims[1]}px | {mode}")
    print(f"   {'存在' if exists else '缺失'}, "
          f"{'>1KB' if size > 1024 else '<=1KB'}")


bar.png: PASS | path=/Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/02-程序设计基础/bar.png | 29761 bytes | 840x504px | RGBA
   存在, >1KB
cloud.png: PASS | path=/Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/02-程序设计基础/cloud.png | 81770 bytes | 900x450px | RGB
   存在, >1KB
